## Configuracion

Unica celda que cada miembro edita.

In [ ]:
NOMBRE          = "Camilo"  # opciones: Jesus, Camilo, Mateo, Sergio

# Ruta a la carpeta del corpus en Drive.
# Si ejecutas !ls /content/drive/MyDrive/ y no ves 'core',
# cambia esta variable con el nombre correcto de la carpeta en tu cuenta.
CARPETA_CORPUS  = "/content/drive/MyDrive/core"

REPO_RAW_URL    = "https://raw.githubusercontent.com/CamiloDS16/nlp-docs-cientificos-es/main/data/particiones.csv"
OUTPUT_PATH     = f"/content/drive/MyDrive/{NOMBRE.lower()}_corpus.parquet"
CHECKPOINT_PATH = f"/content/drive/MyDrive/{NOMBRE.lower()}_corpus_checkpoint.parquet"
CHECKPOINT_CADA = 5000

print(f"Miembro:  {NOMBRE}")
print(f"Corpus:   {CARPETA_CORPUS}")
print(f"Salida:   {OUTPUT_PATH}")

---
## Carga txt

Celdas originales del notebook de Jesus. Montan Drive, verifican la estructura
de carpetas y establecen la conexion con Google Drive API.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive

In [ ]:
import os
carpeta = "/content/drive/MyDrive/core"

In [ ]:
!pip install --upgrade google-api-python-client

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from googleapiclient.discovery import build

service = build('drive', 'v3')

In [ ]:
folder_id = "138g-tMrhRvE2-mjK4VaZlbRScawUNdII"

results = service.files().list(
    q=f"'{folder_id}' in parents and mimeType='text/plain'",
    pageSize=200,
    fields="files(id, name)"
).execute()

files = results.get("files", [])

len(files)

In [ ]:
from googleapiclient.http import MediaIoBaseDownload
import io

textos = []

for file in files:

    request = service.files().get_media(fileId=file['id'])

    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()

    contenido = fh.getvalue().decode("utf-8", errors="ignore")
    textos.append(contenido)

print("Archivos cargados:", len(textos))

In [ ]:
print(textos[0][:2500])

---
## Carga completa filtrada por particion

Las celdas anteriores confirman que Drive esta montado y la conexion funciona.

**Esta seccion tiene dos partes:**

**Parte A — Crear manifest (ejecutar UNA SOLA VEZ, cualquier miembro del equipo)**
Lista todos los archivos del folder, les asigna un indice permanente y guarda
el mapeo como `manifest.csv` en Drive. Una vez creado, todos los miembros
lo usan para saber exactamente que archivo corresponde a cada indice.

**Parte B — Carga individual (cada miembro corre esto con su NOMBRE)**
Carga el manifest, cruza con `particiones.csv`, y descarga solo los archivos
asignados a ese miembro.

In [ ]:
!pip install -q pyarrow pandas tqdm

### Parte A — Crear manifest (UNA SOLA VEZ)

In [ ]:
import pandas as pd

# MANIFEST_PATH es donde se guarda el archivo de mapeo indice -> archivo.
# Debe ser una ubicacion accesible para todos los miembros del equipo.
# Opciones: subcarpeta del Drive compartido, o MyDrive de quien lo cree y luego lo comparte.
MANIFEST_PATH = "/content/drive/MyDrive/corpus_manifest.csv"

# ------------------------------------------------------------------
# SALTAR esta celda si el manifest ya existe.
# Solo necesita correrse UNA VEZ por cualquier miembro del equipo.
# ------------------------------------------------------------------

import os
if os.path.exists(MANIFEST_PATH):
    print(f"El manifest ya existe en: {MANIFEST_PATH}")
    print("Saltate esta celda y continua con la Parte B.")
else:
    print("Creando manifest — lista todos los archivos del folder y asigna indices...")
    print("Esto toma 10-20 minutos. Solo se hace una vez.")
    print()

    todos = []  # (nombre, file_id) en el orden que devuelve la API
    page_token = None
    pagina     = 0

    while True:
        kwargs = dict(
            q=f"'{folder_id}' in parents and mimeType='text/plain' and trashed=false",
            pageSize=1000,
            fields="nextPageToken, files(id, name)",
        )
        if page_token:
            kwargs["pageToken"] = page_token

        res = service.files().list(**kwargs).execute()
        for f in res.get("files", []):
            todos.append((f["name"], f["id"]))

        pagina += 1
        if pagina % 100 == 0:
            print(f"  pagina {pagina} — archivos listados: {len(todos):,}")

        page_token = res.get("nextPageToken")
        if not page_token:
            break

    # construir DataFrame con indice 1-based, nombre y file_id
    df_manifest = pd.DataFrame({
        "index":    range(1, len(todos) + 1),
        "filename": [t[0] for t in todos],
        "file_id":  [t[1] for t in todos],
    })

    df_manifest.to_csv(MANIFEST_PATH, index=False)
    print(f"Manifest creado: {len(df_manifest):,} archivos")
    print(f"Guardado en: {MANIFEST_PATH}")
    print()
    print("Comparte este archivo con todos los miembros del equipo.")
    print("Cada uno debe copiarlo a su MyDrive antes de correr la Parte B.")

### Parte B — Carga individual

In [ ]:
# verificar que el manifest esta disponible
import os

if not os.path.exists(MANIFEST_PATH):
    print(f"Manifest no encontrado en: {MANIFEST_PATH}")
    print("Corre primero la celda de Parte A, o copia el manifest a esa ruta.")
    raise FileNotFoundError(MANIFEST_PATH)

df_manifest = pd.read_csv(MANIFEST_PATH)
print(f"Manifest cargado: {len(df_manifest):,} archivos")
print(df_manifest.head(3).to_string(index=False))

### Cargar particiones

In [ ]:
import pandas as pd

# leer asignaciones directamente desde el repo en GitHub
df_particiones = pd.read_csv(REPO_RAW_URL)

mis_docs = df_particiones[df_particiones["asignado_a"] == NOMBRE].copy()
mis_docs["doc_id"] = mis_docs["doc_id"].astype(str).str.zfill(7)

print(f"Total en particiones.csv:  {len(df_particiones):,}")
print(f"Asignados a {NOMBRE}:       {len(mis_docs):,}")

### Lookup y descarga via API

In [ ]:
from tqdm.notebook import tqdm
from googleapiclient.http import MediaIoBaseDownload
from concurrent.futures import ThreadPoolExecutor, as_completed
import io
import threading

mis_indices  = set(int(doc_id) for doc_id in mis_docs["doc_id"].tolist())
df_asignados = df_manifest[df_manifest["index"].isin(mis_indices)].copy()

print(f"Indices asignados a {NOMBRE}: {len(mis_indices):,}")
print(f"Encontrados en manifest:      {len(df_asignados):,}")
if len(df_asignados) < len(mis_indices):
    print(f"  Advertencia: {len(mis_indices) - len(df_asignados):,} indices no tienen archivo en el manifest")
print()

registros     = []
docs_fallidos = []
_lock         = threading.Lock()


def descargar(fila):
    doc_id = f"{fila.index:07d}"
    try:
        request    = service.files().get_media(fileId=fila.file_id)
        buffer     = io.BytesIO()
        downloader = MediaIoBaseDownload(buffer, request)
        hecho = False
        while not hecho:
            _, hecho = downloader.next_chunk()
        contenido = buffer.getvalue()
        try:
            texto = contenido.decode("utf-8")
        except UnicodeDecodeError:
            texto = contenido.decode("latin-1")
        return {
            "doc_id":     doc_id,
            "filename":   fila.filename,
            "texto":      texto,
            "asignado_a": NOMBRE,
        }
    except Exception as e:
        print(f"  error en {doc_id} ({fila.filename}): {e}")
        return None


filas = list(df_asignados.itertuples())

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(descargar, fila): fila for fila in filas}
    for i, future in enumerate(tqdm(as_completed(futures), total=len(filas), desc=NOMBRE)):
        resultado = future.result()
        if resultado:
            with _lock:
                registros.append(resultado)
        else:
            with _lock:
                docs_fallidos.append(futures[future].index)

        if (i + 1) % CHECKPOINT_CADA == 0:
            with _lock:
                if registros:
                    pd.DataFrame(registros).to_parquet(CHECKPOINT_PATH, index=False)
                    print(f"  checkpoint: {i + 1} documentos procesados")

print(f"\nExitosos: {len(registros):,}")
print(f"Fallidos: {len(docs_fallidos):,}")

### Guardar parquet

In [ ]:
df_corpus = pd.DataFrame(registros)
df_corpus.to_parquet(OUTPUT_PATH, index=False)

tamano_mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)

print(f"Ruta:      {OUTPUT_PATH}")
print(f"Docs:      {len(df_corpus):,}")
print(f"Fallidos:  {len(docs_fallidos):,}")
print(f"Tamano:    {tamano_mb:.1f} MB")

df_corpus.head(2)

### Analisis de calidad (opcional)

In [ ]:
# OPCIONAL
# El siguiente notebook fragmenta documentos en segmentos de 250-1000 palabras.
# Conviene saber cuantos documentos crudos tienen suficiente texto antes de continuar.

df_corpus["num_palabras"] = df_corpus["texto"].str.split().str.len()

print("Distribucion de longitud (palabras por documento):")
print(df_corpus["num_palabras"].describe().round(1).to_string())
print()

docs_ruido  = df_corpus[df_corpus["num_palabras"] < 100]
docs_utiles = df_corpus[df_corpus["num_palabras"] >= 250]

print(f"Docs < 100 palabras (ruido potencial):        {len(docs_ruido):,}  ({len(docs_ruido)/len(df_corpus)*100:.1f}%)")
print(f"Docs >= 250 palabras (aptos para fragmentar): {len(docs_utiles):,}  ({len(docs_utiles)/len(df_corpus)*100:.1f}%)")

### Registro en DVC

In [ ]:
# El parquet no va a GitHub directamente.
# Se copia a data/raw/ y se registra con DVC; solo el archivo .dvc se commitea al repo.

nombre_lower = NOMBRE.lower()

print("Despues de ejecutar este notebook, corre en tu terminal local:")
print()
print(f"  cp {OUTPUT_PATH} data/raw/{nombre_lower}_corpus.parquet")
print(f"  dvc add data/raw/{nombre_lower}_corpus.parquet")
print(f"  git add data/raw/{nombre_lower}_corpus.parquet.dvc .gitignore")
print(f"  git commit -m \"data: add {nombre_lower} partition\"")
print( "  git push")